In [1]:
import sys
from pathlib import Path
from IPython.display import Markdown, HTML

sys.path.append(Path().absolute().parent.as_posix())

from dotenv import load_dotenv
load_dotenv(dotenv_path="../env/.env")

# Import the FEWSConfigAgent class
from talkwithfews.agent import FEWSConfigAgent

In [2]:
# Define your database path
db_path = '../db/fews_config.db3'
log_dir = '/Users/mattijnvanhoek/hkv/fews-networkvisualisation/notebooks'

# Initialize the agent
agent = FEWSConfigAgent(db_path=db_path, log_dir=log_dir)
agent.connect()  # Connect to database and initialize agent

INFO:fews_agent:Using model from PRIMARY_MODEL env: anthropic/claude-3.5-sonnet
INFO:fews_agent:Clearing existing log file: /Users/mattijnvanhoek/hkv/fews-networkvisualisation/notebooks/fews_agent.log
INFO:fews_agent:Logging initialized. Log file: /Users/mattijnvanhoek/hkv/fews-networkvisualisation/notebooks/fews_agent.log
INFO:fews_agent:Initialized FEWSConfigAgent with database: ../db/fews_config.db3
INFO:fews_agent:Using model: anthropic/claude-3.5-sonnet
INFO:fews_agent:Logging to directory: /Users/mattijnvanhoek/hkv/fews-networkvisualisation/notebooks
INFO:fews_agent:Connecting to database: ../db/fews_config.db3
INFO:fews_agent:Extracting LLM-friendly schema information
INFO:fews_agent:Extracting database schema from: ../db/fews_config.db3
INFO:fews_agent:Found 7 tables in the database
INFO:fews_agent:Extracted schema with 7 tables and 8 relationships
INFO:fews_agent:Using comprehensive schema from schema_utils
INFO:fews_agent:Creating FEWS Configuration Agent using anthropic/clau

In [3]:
# Ask for dead points analysis
response = await agent.ask("hoeveel tabbellen zijn er in deze database?")
Markdown(response.data.markdown_text)
Markdown(response.data.markdown_table)
Markdown(response.data.mermaid_diagram)

INFO:fews_agent:Asking agent: hoeveel tabbellen zijn er in deze database?
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:fews_agent:Calling tool: get_database_schema
INFO:fews_agent:Using cached schema information
INFO:fews_agent:Tool get_database_schema completed successfully
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
INFO:fews_agent:Calling tool: execute_query
INFO:fews_agent:{
  "input": "query='SELECT COUNT(*) as aantal_records FROM call_stacks;'"
}
INFO:fews_agent:Tool execute_query completed successfully
INFO:httpx:HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"


graph TD
    ts[ts_sets] --> |1:n| ep[endpoints]
    ts --> |1:n| lnk[links]
    tr[trafos] --> |1:n| lnk
    mc[modins_calls] --> |1:n| tr
    cs[call_stacks] --> |1:n| mc
    ps[props_sets] --> |1:n| cs
    ps --> |1:n| mc

In [ ]:
response

In [4]:
def check_ts_start(mermaid_code):
    lines = mermaid_code.split("\n")
    for i, line in enumerate(lines, start=1):
        if line.strip().startswith("TS"):
            print(f"Line {i} starts with 'TS': {line.strip()}")

# Example usage
mermaid_input = """graph LR
TS[Productie•scalar•WNS2186] --> AGG[Aggregaties Workflow] TS --> WB[Waterbalans Berekeningen] TS --> PE[Peil Evaluaties] TS --> DB[Debiet Berekeningen] TS --> OV[Overige Processen]

AGG --> AGG1[RBP dag]
AGG --> AGG2[poldervierambacht dag]
AGG --> AGG3[binnenpolder dag]
"""

check_ts_start(mermaid_input)

Line 2 starts with 'TS': TS[Productie•scalar•WNS2186] --> AGG[Aggregaties Workflow] TS --> WB[Waterbalans Berekeningen] TS --> PE[Peil Evaluaties] TS --> DB[Debiet Berekeningen] TS --> OV[Overige Processen]


In [5]:
import re

def fix_ts_lines(mermaid_code):
    # Ensure "TS" always starts on a new line
    fixed_code = re.sub(r"(\S+])\s*(TS\s*-->)", r"\1\n\2", mermaid_code)
    return fixed_code

# Example usage
mermaid_input = """graph LR
TS[Productie•scalar•WNS2186] --> AGG[Aggregaties Workflow] TS --> WB[Waterbalans Berekeningen] TS --> PE[Peil Evaluaties] TS --> DB[Debiet Berekeningen] TS --> OV[Overige Processen]

AGG --> AGG1[RBP dag]
AGG --> AGG2[poldervierambacht dag]
AGG --> AGG3[binnenpolder dag]
"""

fixed_mermaid = fix_ts_lines(mermaid_input)
print(fixed_mermaid)


graph LR
TS[Productie•scalar•WNS2186] --> AGG[Aggregaties Workflow]
TS --> WB[Waterbalans Berekeningen]
TS --> PE[Peil Evaluaties]
TS --> DB[Debiet Berekeningen]
TS --> OV[Overige Processen]

AGG --> AGG1[RBP dag]
AGG --> AGG2[poldervierambacht dag]
AGG --> AGG3[binnenpolder dag]



In [ ]:
agent.close()

In [9]:
import re

def fix_mermaid_syntax(mermaid_code):
    # Split the input into lines
    lines = mermaid_code.strip().split("\n")
    
    # Get the graph type from the first line (e.g., "graph TD" or "graph LR")
    first_line = lines[0].strip()
    
    # Ensure the first line contains only "graph <type>"
    if " " in first_line:
        first_line = first_line.split(" ")[0]  # Keep only "graph TD" or similar
    
    # Join the rest of the lines into one string to handle multi-line edges and subgraphs
    rest_of_code = " ".join(lines[1:])
    
    # Ensure multiline node labels (e.g., "WNS2186Tijdreeks\n10-minuut waarden") are properly formatted
    rest_of_code = re.sub(r'(\S+)\["([^"]+?)"\]', r'\1["\2"]', rest_of_code)  # Ensure correct node format
    
    # Handle subgraph blocks (e.g., subgraph Aggregaties ... end) properly
    rest_of_code = re.sub(r"(subgraph.*?end)", lambda m: m.group(0).replace('\n', ' ').strip(), rest_of_code)

    # Ensure each connection is on a separate line
    rest_of_code = re.sub(r"(\S+)\s*-->\s*(\S+)", r"\1 --> \2", rest_of_code)  # Correct edge formatting
    rest_of_code = re.sub(r"\s*(\S+\s*-->\s*\S+)\s*", r"\1\n", rest_of_code)  # Place each edge on a new line
    
    # Reassemble the fixed Mermaid code
    fixed_mermaid = f"{first_line}\n\n{rest_of_code.strip()}"
    
    return fixed_mermaid

# Example Mermaid input
mermaid_input = """graph TD WNS2186[WNS2186 Tijdreeks] --> Aggregaties WNS2186 --> Peilbeheer WNS2186 --> Waterbalans WNS2186 --> Procesmonitoring WNS2186 --> Debiet

Aggregaties --> RBP[RBP dag]
Aggregaties --> Polder[Poldervierambacht dag]
Aggregaties --> Binnen[Binnenpolder dag]

Peilbeheer --> Gebieds[Gebiedsgemiddelden]
Peilbeheer --> Praktijk[Praktijkpeil]
Peilbeheer --> Evaluatie[Peilevaluatie]

Waterbalans --> WBPolder[Waterbalans polder]
Waterbalans --> WBPraktijk[Waterbalans praktijkpeilgebied]

Procesmonitoring --> Overzicht[Procesoverzicht]
Procesmonitoring --> Vulling[Vullingsgraad]

Debiet --> Stuwen[Debieten stuwen]
Debiet --> Pomp[Pompvolumes]
"""

# Apply fix
fixed_mermaid = fix_mermaid_syntax(mermaid_input)
print(fixed_mermaid)


graph

Aggregaties --> RBP[RBP
dag]Aggregaties --> Polder[Poldervierambacht
dag]Aggregaties --> Binnen[Binnenpolder
dag]Peilbeheer --> Gebieds[Gebiedsgemiddelden]
Peilbeheer --> Praktijk[Praktijkpeil]
Peilbeheer --> Evaluatie[Peilevaluatie]
Waterbalans --> WBPolder[Waterbalans
polder]Waterbalans --> WBPraktijk[Waterbalans
praktijkpeilgebied]Procesmonitoring --> Overzicht[Procesoverzicht]
Procesmonitoring --> Vulling[Vullingsgraad]
Debiet --> Stuwen[Debieten
stuwen]Debiet --> Pomp[Pompvolumes]


In [ ]:
import re

def fix_mermaid_syntax(mermaid_code):
    # Split the input by lines
    lines = mermaid_code.strip().split("\n")
    
    # Get the graph type from the first line (e.g., "graph TD" or "graph LR")
    first_line = lines[0].strip()
    
    # Ensure the first line contains only "graph <type>"
    if " " in first_line:
        first_line = first_line.split(" ")[0]  # Keep only "graph TD" or similar
    
    # Join the rest of the lines into one string to handle multi-line edges and subgraphs
    rest_of_code = " ".join(lines[1:])
    
    # Handle multiline node labels by ensuring new lines inside labels are respected
    rest_of_code = re.sub(r'(\S+)\["([^"]+)"\]', r'\1["\2"]', rest_of_code)  # Ensure correct node format
    
    # Split each connection onto a new line properly
    rest_of_code = re.sub(r"(\S+)\s*-->\s*(\S+)", r"\1 --> \2", rest_of_code)  # Correct spacing for edges
    rest_of_code = re.sub(r"\s*(\S+\s*-->\s*\S+)\s*", r"\1\n", rest_of_code)  # Ensure each connection is on a new line
    
    # Handle subgraphs by ensuring they're properly spaced and formatted
    rest_of_code = re.sub(r"(subgraph.*?end)", lambda m: m.group(0).replace('\n', ' ').strip(), rest_of_code)

    # Reassemble the fixed Mermaid code
    fixed_mermaid = f"{first_line}\n\n{rest_of_code.strip()}"
    
    return fixed_mermaid


# Example Mermaid input (with subgraphs and multiline labels)
mermaid_input = """graph LR WNS2186["WNS2186 Meetreeks\n10-minuut waarden"]

subgraph Aggregaties
AGG["Dagelijkse\nAggregaties"]
end

subgraph "Waterbalans"
WB["Waterbalans\nBerekeningen"]
end

subgraph "Peil Evaluatie"
PE["Peil Evaluaties"]
end

subgraph "Debieten"
DEB["Debiet\nBerekeningen"]
end

subgraph "Proces Overzicht"
PO["Proces\nOverzichten"]
end

subgraph "Vullingsgraad"
VG["Vullingsgraad\nBepaling"]
end

WNS2186 --> AGG
WNS2186 --> WB
WNS2186 --> PE
WNS2186 --> DEB
WNS2186 --> PO
WNS2186 --> VG
"""

# Apply the fix
fixed_mermaid = fix_mermaid_syntax(mermaid_input)
print(fixed_mermaid)
